# K-Nearest Neighbors Classifier

## Load Preprocessed Data

In [ ]:
from preprocessing2 import preprocess 
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="pca")


## Model Definition

In [ ]:
import numpy as np
class KNN:
    def __init__(self, k=3,  metric='euclidean', weights='uniform'):
        self.k = k
        self.metric = metric
        self.weights = weights
    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
    def _compute_distances(self, X):
        """Compute distances from all test points to all training points at once."""
        if self.metric == 'euclidean':
            a2 = np.sum(X ** 2, axis=1, keepdims=True)           
            b2 = np.sum(self.X_train ** 2, axis=1, keepdims=True) 
            dists = np.sqrt(np.maximum(a2 + b2.T - 2 * X @ self.X_train.T, 0))

        else:
            raise ValueError(f"Metric '{self.metric}' not supported in vectorized mode")

        return dists 
         
            
    def predict(self, X):
        X = np.array(X)
        dists = self._compute_distances(X)          # (n_test, n_train)
        k_idx = np.argsort(dists, axis=1)[:, :self.k]  # (n_test, k)
        #k_idx = np.argpartition(dists, self.k, axis=1)[:, :self.k]  # (n_test, k)


        predictions = []
        for i, neighbors in enumerate(k_idx):
            labels = self.y_train[neighbors]
            if self.weights == 'uniform':
                weights = np.ones(self.k)
            else:
                weights = 1 / (dists[i, neighbors] + 1e-9)

            
            label_counts = {}
            for label, w in zip(labels, weights):
                label_counts[label] = label_counts.get(label, 0) + w
            predicted_label = max(label_counts, key=label_counts.get)
            predictions.append(predicted_label)

        return np.array(predictions)


## Training

In [ ]:
#creata an instance of the KNN class
knn = KNN(k=3, metric="euclidean", weights="uniform")
#fit the model to the training data
knn.fit(X_train, y_train)



## Elbow Method Helper

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_elbow_knn_manual(knn, X_val, y_val, k_range):
    errors = []

    for k in k_range:
        print(f"Running k = {k} ...")

        knn.k = k  # set k

        y_pred = knn.predict(X_val)

        acc = np.mean(y_pred == y_val)  
        error = 1 - acc

        errors.append(error)

        print(f"k={k}, Accuracy={acc:.4f}, Error={error:.4f}")

    # Plot
    plt.figure()
    plt.plot(k_range, errors, marker='o')
    plt.xlabel("k")
    plt.ylabel("Error")
    plt.title("Elbow Method (Manual KNN)")
    plt.show()

    return errors

## Choosing k via Elbow Method

In [ ]:
k_values = range(1, 15, 2)

errors = plot_elbow_knn_manual(knn, X_val, y_val, k_values)
print("Done running elbow method for KNN.")
print("Errors for KNN:", errors)

## Evaluation Helper

In [ ]:
def evaluate_model(X, y, model, dataset_name="Validation"):
    """
    Evaluate multiclass model (digits 0-9)
    """

    y_pred = model.predict(X)
    y_true = y

    n_classes = 10  

    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1

    accuracy = np.trace(cm) / np.sum(cm)

    precision = np.zeros(n_classes)
    recall = np.zeros(n_classes)
    f1 = np.zeros(n_classes)

    for i in range(n_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp

        precision[i] = tp / (tp + fp + 1e-9)
        recall[i]    = tp / (tp + fn + 1e-9)
        f1[i]        = 2 * precision[i] * recall[i] / (precision[i] + recall[i] + 1e-9)

    macro_f1 = np.mean(f1)

    print(f"\n--- {dataset_name} Results ---")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Macro F1  : {macro_f1:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(f"{'Class':<10}{'Precision':>10}{'Recall':>10}{'F1-score':>12}")

    for i in range(n_classes):
        print(f"{i:<10}{precision[i]:>10.2f}{recall[i]:>10.2f}{f1[i]:>12.2f}")

## Cross-Validation

In [ ]:
#cross validate on the validation set
import numpy as np

# Different feature extraction methods
feature_methods = ["cnn", "pca", "flatten","hog"]

# Hyperparameters to search
param_grid = {
    "k": [3, 5],
    "metric": ["euclidean"],
    "weights": ["uniform", "distance"]
}

best_f1_score = 0
best_params = {}

total_runs = (
    len(feature_methods)
    * len(param_grid["k"])
    * len(param_grid["metric"])
    * len(param_grid["weights"])
)

current_run = 1

for feature_method in feature_methods:

    # Load and preprocess data
    X_train, y_train, X_val, y_val, X_test, y_test, _ = preprocess(
        feature_method=feature_method,
        n_pca=50
    )

    # Create folds
    folds = k_fold_indices(X_train, k=3)

    # Hyperparameter search
    for k in param_grid["k"]:
        for metric in param_grid["metric"]:
            for weights in param_grid["weights"]:

                print(
                    f"--- Run {current_run}/{total_runs} | "
                    f"Feature:{feature_method} | "
                    f"k:{k} | metric:{metric} | weights:{weights} ---"
                )

                fold_f1_scores = []

                for train_idx, val_idx in folds:

                    X_fold_train = X_train[train_idx]
                    y_fold_train = y_train[train_idx]

                    X_fold_val = X_train[val_idx]
                    y_fold_val = y_train[val_idx]

                    # Create fresh KNN model
                    cv_model = KNN(
                        k=k,
                        metric=metric,
                        weights=weights
                    )

                    cv_model.fit(X_fold_train, y_fold_train)

                    preds = cv_model.predict(X_fold_val)

                    fold_f1 = custom_macro_f1_score(
                        y_fold_val,
                        preds,
                        n_classes=10
                    )

                    fold_f1_scores.append(fold_f1)

                avg_f1 = np.mean(fold_f1_scores)

                print(f"    -> 3-Fold Average Macro F1: {avg_f1:.4f}\n")

                # Save best parameters
                if avg_f1 > best_f1_score:

                    best_f1_score = avg_f1

                    best_params = {
                        "feature_method": feature_method,
                        "k": k,
                        "metric": metric,
                        "weights": weights
                    }

                current_run += 1

print("=" * 50)
print("GRID SEARCH COMPLETE")
print("=" * 50)

print(f"Best CV Macro F1: {best_f1_score:.4f}")
print(f"Best Parameters: {best_params}")

## Evaluation on Validation Set

In [ ]:
evaluate_model(X_val, y_val, knn, dataset_name="Validation FULL")

## Evaluation on Test Set

In [ ]:
evaluate_model(X_test, y_test, knn, dataset_name="Validation FULL")